***PROYECTO TERCIO 2: DIAGNOSTICO DE CÁNCER DE MAMA***

Para nuestro proyecto decidimos utilizar el problema del diagnostico de cancer de mama donde a partir de los datos determinar si es tumor benigno o maligno, el problema es importante ya que gracias a esto  podemos demostrar como con los datos podemos entender y poder pronosticar tempranamente si un tumor o no es maligno.

Nuestro problema es de clasificación, ya que claramente es un problema binario, es o no es, posterior a eso  vamos primero a analizar el dataset


In [13]:
#LIBRERIAS NE CESARIAS
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers

#LOS NOMBRES FUERONS SACADOS DE EL ARCHIVO NAMES
nombres_columnas = ['id_paciente', 'clump_thickness', 'uniformity_cell_size', 'uniformity_cell_shape', 'marginal_adhesion', 'single_epithelial_cell_size',
 'bare_nuclei', 'bland_chromatin', 'normal_nucleoli', 'mitoses', 'clase']
data_set_cancer = pd.read_csv('data/breast-cancer-wisconsin.data', header=None, names=nombres_columnas, na_values='?')

print(data_set_cancer.head())
data_set_cancer.info()
data_set_cancer.describe()

   id_paciente  clump_thickness  uniformity_cell_size  uniformity_cell_shape  \
0      1000025                5                     1                      1   
1      1002945                5                     4                      4   
2      1015425                3                     1                      1   
3      1016277                6                     8                      8   
4      1017023                4                     1                      1   

   marginal_adhesion  single_epithelial_cell_size  bare_nuclei  \
0                  1                            2          1.0   
1                  5                            7         10.0   
2                  1                            2          2.0   
3                  1                            3          4.0   
4                  3                            2          1.0   

   bland_chromatin  normal_nucleoli  mitoses  clase  
0                3                1        1      2  
1             

,id_paciente,clump_thickness,uniformity_cell_size,uniformity_cell_shape,marginal_adhesion,single_epithelial_cell_size,bare_nuclei,bland_chromatin,normal_nucleoli,mitoses,clase
count,6.990000e+02,699.000000,699.000000,699.000000,699.000000,699.000000,683.000000,699.000000,699.000000,699.000000,699.000000
mean,1.071704e+06,4.417740,3.134478,3.207439,2.806867,3.216023,3.544656,3.437768,2.866953,1.589413,2.689557
std,6.170957e+05,2.815741,3.051459,2.971913,2.855379,2.214300,3.643857,2.438364,3.053634,1.715078,0.951273
min,6.163400e+04,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,2.000000
25%,8.706885e+05,2.000000,1.000000,1.000000,1.000000,2.000000,1.000000,2.000000,1.000000,1.000000,2.000000
50%,1.171710e+06,4.000000,1.000000,1.000000,1.000000,2.000000,1.000000,3.000000,1.000000,1.000000,2.000000
75%,1.238298e+06,6.000000,5.000000,5.000000,4.000000,4.000000,6.000000,5.000000,4.000000,1.000000,4.000000
max,1.345435e+07,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,4.000000


el data set tiene 9 entradas o  columnas y una salida: las columnas coresponden a los siguientes informacion:

id_paciente: Es el número de identificación único de la muestra médica de cada paciente. Nosotros lo eliminaremos del análisis porque no aporta ningún patrón matemático para predecir el cáncer.

clump_thickness (Grosor del agrupamiento): Evalúa si las células están apiladas en múltiples capas (común en células malignas) o agrupadas en una sola capa ordenada (común en células benignas).

uniformity_cell_size (Uniformidad del tamaño celular): Nos indica qué tan consistentes son las células en su tamaño. En un tejido sano, las células son casi del mismo tamaño; en el cáncer, varían drásticamente.

uniformity_cell_shape (Uniformidad de la forma celular): Funciona igual que la anterior, pero evaluando la forma. Las células cancerosas tienden a tener contornos irregulares y deformes.

marginal_adhesion (Adhesión marginal): Mide qué tan bien se "pegan" o adhieren las células entre sí. Las células normales suelen mantenerse unidas, mientras que las células malignas pierden esta cohesión y tienden a separarse.

single_epithelial_cell_size (Tamaño de célula epitelial individual): Mide el tamaño de un tipo específico de célula (epitelial). Un agrandamiento anormal en estas células es una señal temprana de que el tejido se está volviendo maligno.

bare_nuclei (Núcleos desnudos): Cuenta los núcleos celulares que se han quedado sin su capa exterior (citoplasma). Nota importante para nosotros

bland_chromatin (Cromatina blanda/uniforme): Describe la textura del material genético (ADN) dentro del núcleo. En células sanas, la textura se ve uniforme y "suave"; en las cancerosas, se agruma y se ve gruesa o áspera.

normal_nucleoli (Nucléolos normales): Los nucléolos son estructuras diminutas dentro del núcleo. En células sanas son apenas visibles, pero en células cancerosas se inflaman y se vuelven muy notorios.

mitoses (Mitosis): Nos indica la tasa de división y reproducción celular. Un nivel alto de mitosis refleja que las células se están multiplicando agresivamente y sin control, una característica principal del cáncer.

clase: Es nuestra variable objetivo (lo que queremos predecir). Como vimos en el manual, usa el valor 2 para los tumores benignos y 4 para los malignos, los cuales nosotros cambiaremos a 0 y 1 para nuestro perceptrón.

entonces las 9 primeras van a ser de entrada mientras la columna clave es nuestro obejtivo o salida.

para el perceptron vamos a usar las siguientes funciones:
ACTIVACION = FUNCION SIGMOIDE
esa función nos sirve para que cualquier dato que se ingrese este entre 0 y 1 asi si por ejemplo si nos da 0.85 podriamos decir que tiene un 85% de que va a presentar un cancer maligno

PERDIDA = ENTROPIA BINARIA CRUZADA
esta funcion "castiga" o genera un error grande cuando se diga por ejemplo que tiene un cancer maligno (0.98) y en efecto tenga un cancer malgino nos dara un error muy bajo mientras que si decimos que no es maligno (0.01) entonces tendra un error muy grande por lo tanto nos sirve mucho para este caso

In [12]:
print(data_set_cancer.isnull().sum())

id_paciente                     0
clump_thickness                 0
uniformity_cell_size            0
uniformity_cell_shape           0
marginal_adhesion               0
single_epithelial_cell_size     0
bare_nuclei                    16
bland_chromatin                 0
normal_nucleoli                 0
mitoses                         0
clase                           0
dtype: int64
